# ⑫ o2pulse_flattening — AI 과제 `o2p_ai_v2` (Keras 노트북)
v1(`o2p_ai_v1`, PyTorch 스크립트)과 다른 점 — 사용자 결정 2026-10-22: **A2 딥러닝을 Keras(TensorFlow)로**, 파일을 **주피터 노트북**으로. 입력 per_test를 `results_o2p_v3/per_test.csv`로.
구조·하이퍼파라미터·분할·seed는 v1 그대로(인과 dilated CNN 32필터 kernel 5 dilation 1/2/4/8 · GRU 64 · Adam 1e-3 · 전역 gradient clip 1.0 · batch 16 · 최대 150 epoch · patience 15 · 마스크 MSE).
A1 · A3 · A4 · A2 Ridge는 같은 코드·같은 seed·같은 입력 열이라 v1과 같아야 한다 → `v1_vs_v2.csv`에서 assert. A2 CNN/GRU는 프레임워크가 달라 초기값·난수가 달라진다 → 차이를 표로만 남긴다.

사전지정: 03_RESEARCH/projects/o2pulse_flattening/analysis.md 'AI 과제 사전지정' (2026-10-22 승인 1, AI) · 계획: 05_RUNS/o2pulse_flattening/2026-10-22_ai_plan.md
A1 평탄화 조기 예측 · A2 가스분석 없는 VO2/O2P 시계열 추정(딥러닝) · A3 최대 O2P 회귀 · A4 O2P 곡선 군집.
실행: 이 노트북을 자기 폴더(04_OUTPUTS/code_o2p_ai_v2)에서 Restart & Run All. 경로는 환경변수 O2P_RAW · O2P_PERTEST · O2P_OUT로 바꿀 수 있다(OUT가 이미 있으면 멈춘다).
한계(독립 검증 2026-10-22): A1의 절단 시점 t_end − 120 s와 elapsed_c는 운동 종료 시각(t_end)을 사후에 알아야 정해진다 — 실시간 예측이 아니다.
A2 릿지 alpha = 1.0 고정(튜닝 없음). 사후(○): A4 k 1–12 BIC 민감도 · 램프 속도별 평탄화 비율 → posthoc_ai.csv (프로토콜 층화의 사전지정 분석은 ⑫ o2p_v3 SE8–SE10).
방법: [핸즈온1 p.86–92, 101–103, 120–136, 148–163, 268–295, 325–362, 384–429] · [핸즈온2 p.176–180, 191–194, 199–203, 266] · 해석 점검: [해석학 p.68–72, 155–161, 191–194] · 수학 근거: [수학 p.237–240]

In [ ]:
import os, sys, json, hashlib, platform, datetime, math, time, warnings
from importlib import metadata
import numpy as np, pandas as pd
from scipy import stats
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import roc_auc_score, brier_score_loss, mean_absolute_error, r2_score, cohen_kappa_score
from sklearn.inspection import permutation_importance
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.stats import loguniform
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import keras, tensorflow as tf
from keras import layers, ops
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = None
warnings.filterwarnings('ignore', category=UserWarning); warnings.filterwarnings('ignore', category=RuntimeWarning)
from sklearn.exceptions import ConvergenceWarning; warnings.filterwarnings('ignore', category=ConvergenceWarning)

H = os.getcwd()   # 노트북 폴더(04_OUTPUTS/code_o2p_ai_v2)
RAW = os.environ.get('O2P_RAW') or os.path.join(H, '..', 'raw')   # 공개 저장소판: 저장소 루트의 raw/ (PhysioNet 원자료는 저장소에 없음)
PERTEST = os.environ.get('O2P_PERTEST') or os.path.join(H, '..', 'results_o2p_v3', 'per_test.csv')
OUT = os.environ.get('O2P_OUT') or os.path.join(H, '..', 'results_o2p_ai_v2')
V1_RES = os.environ.get('O2P_V1') or os.path.join(H, '..', 'results_o2p_ai_v1')   # v1 대조(없으면 대조 생략)
FAST = os.environ.get('O2P_FAST') == '1'          # 디버그 전용(결과로 쓰지 않음)
CACHE = os.environ.get('O2P_CACHE')                 # 컨테이너 재시작 대비 fold 단위 체크포인트(결과 동일 — 같은 seed · 같은 입력)
import joblib
if CACHE: os.makedirs(CACHE, exist_ok=True)
def cget(key):
    p = os.path.join(CACHE, key + '.pkl') if CACHE else None
    return joblib.load(p) if p and os.path.exists(p) else None
def cput(key, obj):
    if CACHE: joblib.dump(obj, os.path.join(CACHE, key + '.pkl'))
assert not os.path.exists(OUT), f'결과 폴더가 이미 있다: {OUT}'
os.makedirs(OUT); FIG = os.path.join(OUT, 'figures'); os.makedirs(FIG)
SEED, JSEED, NB = 20261022, 20261023, 2000
REPEATS, KOUT, KIN, NITER = (1, 5, 3, 3) if FAST else (5, 5, 3, 20)
SHA = {'subject-info.csv': 'e9ca5ba20eb1272df56c11d3e584ec3997d5484454812bd94966e58082415f16',
       'test_measure.csv': '3b5ed5586dc7412ccdfb26f804e29c06482a2a1df2060399c2731ebc68d4613d'}
VALID = {'vo2peak_kg': (15, 90), 'hrmax': (120, 230), 'o2p_max': (5, 45), 'rer_peak': (0.9, 1.6)}
PRESPEC = ['A1', 'A2', 'A3', 'A4', 'SG', 'PI', 'O', 'D']
assert PRESPEC == ['A1', 'A2', 'A3', 'A4', 'SG', 'PI', 'O', 'D'], '사전지정 ↔ plan 불일치'
G_NAMES = ['G1_flow', 'G2_engine', 'G3_primary', 'G4_subgroups', 'G5_compare', 'G6_importance', 'G7_a2_inputs', 'G8_curves', 'G9_residuals', 'G10_outliers']
EXPECTED_OUTPUTS = ['run_info.json', 'requirements_exact.txt', 'engine_check.csv', 'sample_flow.csv', 'A1_folds.csv', 'A1_summary.csv', 'A1_compare.csv',
                    'A2_summary.csv', 'A2_per_test.csv', 'A3_folds.csv', 'A3_summary.csv', 'A3_compare.csv', 'A3_oof.csv', 'A4_bic.csv',
                    'A4_clusters.csv', 'A4_repeat.csv', 'posthoc_ai.csv', 'v1_vs_v2.csv', 'subgroup.csv', 'importance.csv', 'outliers.csv', 'decisions.json', 'features_A1_A3.csv'] + \
                   [f'figures/{g}.png' for g in G_NAMES] + [f'figures/{g}_data.csv' for g in ['G3', 'G8', 'G9', 'G10']]
Z = stats.norm.ppf(.975)
keras.utils.set_random_seed(SEED); tf.config.experimental.enable_op_determinism()   # 같은 기계·같은 버전에서 재실행 동일

def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
for fn, ex in SHA.items(): assert sha(os.path.join(RAW, fn)) == ex, f'SHA-256 불일치 {fn}'
PT_SHA = sha(PERTEST)
s = pd.read_csv(os.path.join(RAW, 'subject-info.csv'))
m = pd.read_csv(os.path.join(RAW, 'test_measure.csv')).sort_values(['ID_test', 'time'], kind='stable')
T = pd.read_csv(PERTEST)
assert len(T) == 992 and set(T.id_test) == set(s.ID_test)
assert set(T.loc[T.fail.fillna('') != '', 'id_test']) >= {'218_1', '741_1', '539_1'}, 'o2p_v2/v3 제외가 반영된 per_test가 아니다'
assert 'ramp_last120' in T, 'o2p_v3 per_test가 아니다'
A = T[(T.fail.fillna('') == '') & (T.age >= 18) & (T.main_evaluable == True)].copy().reset_index(drop=True)
A['F'] = A.main_F.astype(int); info = s.set_index('ID_test')
A['height'] = [float(info.at[t, 'Height']) for t in A.id_test]

In [ ]:
# ── 0. 엔진 검증 ─────────────────────────────────────────────
def nb_ci(d, n_test, n_train):
    d = np.asarray(d, float); J = len(d); v = np.var(d, ddof=1) * (1 / J + n_test / n_train)
    se = math.sqrt(v); tq = stats.t.ppf(.975, J - 1); tv = d.mean() / se if se > 0 else np.inf
    return d.mean(), d.mean() - tq * se, d.mean() + tq * se, 2 * stats.t.sf(abs(tv), J - 1)
def auc_manual(y, p):
    y = np.asarray(y); p = np.asarray(p); r = stats.rankdata(p); n1 = y.sum(); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
def kappa(a, b):
    a, b = np.asarray(a), np.asarray(b); cats = np.union1d(a, b); po = np.mean(a == b)
    pe = sum(np.mean(a == c) * np.mean(b == c) for c in cats); return np.nan if pe >= 1 else (po - pe) / (1 - pe)
rng = np.random.default_rng(SEED); eng = []
for i in range(5):
    y = (rng.random(200) < .4).astype(int); p = rng.random(200) + .3 * y
    eng.append(dict(check=f'auc sim{i}', a=auc_manual(y, p), b=roc_auc_score(y, p)))
    a_ = rng.integers(0, 3, 150); b_ = np.where(rng.random(150) < .6, a_, rng.integers(0, 3, 150))
    eng.append(dict(check=f'kappa3 sim{i}', a=kappa(a_, b_), b=cohen_kappa_score(a_, b_)))
d_ = np.array([.02, .05, -.01, .03, .04]); mu, lo, hi, _ = nb_ci(d_, 20, 80)
se_ = math.sqrt(np.var(d_, ddof=1) * (1 / 5 + 20 / 80)); eng.append(dict(check='nadeau-bengio lo', a=lo, b=d_.mean() - stats.t.ppf(.975, 4) * se_))
E = pd.DataFrame(eng); E['absdiff'] = (E.a - E.b).abs(); assert (E.absdiff < 1e-10).all(), E

In [ ]:
# ── 1. 특징 만들기 ───────────────────────────────────────────
def binned(t, y, a, b, w):
    k = (t >= a) & (t < b) & ~np.isnan(y)
    if k.sum() == 0: return np.array([]), np.array([])
    d = pd.DataFrame(dict(b=np.floor((t[k] - a) / w).astype(int), y=y[k], t=t[k])).groupby('b').mean()
    return d.t.to_numpy(), d.y.to_numpy()
def slope(t, y):
    k = ~np.isnan(y); return np.polyfit(t[k] / 60, y[k], 1)[0] if k.sum() >= 3 else np.nan
def mean_in(t, y, a, b):
    k = (t >= a) & (t < b) & ~np.isnan(y); return float(np.mean(y[k])) if k.sum() else np.nan
FEAT = {}; SEQ = {}
for tid, g in m[m.ID_test.isin(A.id_test)].groupby('ID_test'):
    g = g[g.HR.between(30, 230) | g.HR.isna()]
    t = (g.time - g.time.min()).to_numpy(float); hr = g.HR.to_numpy(float); sp = g.Speed.to_numpy(float)
    vo2 = g.VO2.to_numpy(float); vco2 = g.VCO2.to_numpy(float); ve = g.VE.to_numpy(float); rr = g.RR.to_numpy(float)
    r = A.loc[A.id_test == tid].iloc[0]; te = r.t_end; c = te - 120; age = r.age
    f = {}
    for nm, y in [('hr', hr), ('vo2', vo2), ('vco2', vco2), ('ve', ve), ('rr', rr), ('speed', sp)]: f[f'{nm}_c'] = mean_in(t, y, c - 30, c)
    f['o2p_c'] = f['vo2_c'] / f['hr_c']; f['rer_c'] = f['vco2_c'] / f['vo2_c']; f['vevco2_c'] = f['ve_c'] / f['vco2_c']; f['vevo2_c'] = f['ve_c'] / f['vo2_c']
    for nm, y in [('hr', hr), ('vo2', vo2), ('ve', ve), ('speed', sp)]:
        tb, yb = binned(t, y, c - 120, c, 10); f[f'{nm}_slope'] = slope(tb, yb)
    tb, vb = binned(t, vo2, c - 120, c, 10); _, hb = binned(t, hr, c - 120, c, 10)
    f['o2p_slope'] = slope(tb, vb / hb) if len(vb) == len(hb) and len(vb) else np.nan
    f['elapsed_c'] = c / 60; f['phr_c'] = 100 * f['hr_c'] / (208 - .7 * age)
    # A3(가스 없음): 상승 구간 HR–속도
    ip = int(np.nanargmax(sp)); up = t < t[ip]
    hs = pd.Series(hr).rolling(10, min_periods=3, center=True).mean().to_numpy()
    for v in (10, 12):
        k = up & (np.abs(sp - v) <= 1) & ~np.isnan(hs)
        f[f'hr_at{v}'] = float(np.polyval(np.polyfit(sp[k], hs[k], 1), v)) if k.sum() >= 5 and np.ptp(sp[k]) >= 1 else np.nan
    k = up & (sp >= 8) & (sp <= 16) & ~np.isnan(hs); f['hr_speed_slope'] = np.polyfit(sp[k], hs[k], 1)[0] if k.sum() >= 5 and np.ptp(sp[k]) >= 2 else np.nan
    f['hr_warm'] = mean_in(t, hr, 60, 120); f['test_min'] = te / 60
    FEAT[tid] = f
    # A2 시퀀스: 운동 구간 5 s 격자
    ex = t < te; nb_ = int(np.ceil(te / 5))
    def grid(y):
        d = pd.DataFrame(dict(b=(t[ex] // 5).astype(int), y=y[ex])).groupby('b').y.mean().reindex(range(nb_))
        return d.interpolate(limit_direction='both').to_numpy()
    SEQ[tid] = dict(hr=grid(hr), sp=grid(sp), rr=grid(rr), ve=grid(ve), vo2=grid(vo2), te=te)
FE = pd.DataFrame.from_dict(FEAT, orient='index'); FE.index.name = 'id_test'
A = A.merge(FE.reset_index(), on='id_test', how='left')
A1_X = ['age', 'sex', 'weight', 'height', 'hr_c', 'vo2_c', 'vco2_c', 've_c', 'rr_c', 'speed_c', 'o2p_c', 'rer_c', 'vevco2_c', 'vevo2_c',
        'hr_slope', 'vo2_slope', 've_slope', 'speed_slope', 'o2p_slope', 'elapsed_c', 'phr_c']
A3_X = ['age', 'sex', 'weight', 'height', 'hrmax', 'speed_max', 'test_min', 'hr_at10', 'hr_at12', 'hr_speed_slope', 'hr_warm']
A[['id_test', 'subj', 'F', 'o2p_max'] + A1_X + [x for x in A3_X if x not in A1_X]].to_csv(os.path.join(OUT, 'features_A1_A3.csv'), index=False)  # 로컬 전용(DUA)

In [ ]:
# ── 2. 참가자 단위 바깥 분할 ─────────────────────────────────
def outer_splits(groups, repeats=REPEATS, k=KOUT, seed=SEED):
    ug = np.unique(groups); out = []
    for r in range(repeats):
        perm = np.random.default_rng(seed + r).permutation(ug)
        for j, fold in enumerate(np.array_split(perm, k)):
            te_ = np.isin(groups, fold); assert not set(groups[te_]) & set(groups[~te_]), '참가자 누수'
            out.append((r, j, np.where(~te_)[0], np.where(te_)[0]))
    return out
G = A.subj.to_numpy(); SPL = outer_splits(G)
eng.append(dict(check='outer split participant overlap (count)', a=0.0, b=0.0)); E = pd.DataFrame(eng); E['absdiff'] = (E.a - E.b).abs()

def pre(scale=True): return [('imp', SimpleImputer(strategy='median'))] + ([('sc', StandardScaler())] if scale else [])
CLF = {'LR': (Pipeline(pre() + [('m', LogisticRegression(max_iter=5000))]), {'m__C': loguniform(1e-3, 1e2)}),
       'RF': (Pipeline(pre(False) + [('m', RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=2))]),
              {'m__max_depth': [None, 3, 5, 8], 'm__min_samples_leaf': [1, 5, 10, 20], 'm__max_features': ['sqrt', .5, 1.0]}),
       'HGB': (HistGradientBoostingClassifier(random_state=SEED),
               {'learning_rate': loguniform(.01, .3), 'max_depth': [None, 3, 5], 'max_leaf_nodes': [7, 15, 31], 'min_samples_leaf': [10, 20, 40], 'l2_regularization': [0, .1, 1]}),
       'MLP': (Pipeline(pre() + [('m', MLPClassifier(max_iter=2000, early_stopping=True, random_state=SEED))]),
               {'m__hidden_layer_sizes': [(16,), (32,), (32, 16), (64, 32)], 'm__alpha': loguniform(1e-4, 1), 'm__learning_rate_init': loguniform(1e-4, 1e-2)})}
REG = {'Linear': (Pipeline(pre() + [('m', LinearRegression())]), None),
       'Ridge': (Pipeline(pre() + [('m', Ridge())]), {'m__alpha': loguniform(1e-3, 1e3)}),
       'RF': (Pipeline(pre(False) + [('m', RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=2))]),
              {'m__max_depth': [None, 3, 5, 8], 'm__min_samples_leaf': [1, 5, 10, 20], 'm__max_features': ['sqrt', .5, 1.0]}),
       'HGB': (HistGradientBoostingRegressor(random_state=SEED),
               {'learning_rate': loguniform(.01, .3), 'max_depth': [None, 3, 5], 'max_leaf_nodes': [7, 15, 31], 'min_samples_leaf': [10, 20, 40], 'l2_regularization': [0, .1, 1]}),
       'MLP': (Pipeline(pre() + [('m', MLPRegressor(max_iter=3000, early_stopping=True, random_state=SEED))]),
               {'m__hidden_layer_sizes': [(16,), (32,), (32, 16), (64, 32)], 'm__alpha': loguniform(1e-4, 1), 'm__learning_rate_init': loguniform(1e-4, 1e-2)})}

def nested(task, X, y, models, scoring):
    rows, oof, best_models = [], [], {}
    for (r, j, tr, te_) in SPL:
        hit = cget(f'{task}_{r}_{j}')
        if hit is not None:
            rows += hit[0]; oof += hit[1]
            for nm, mdl in hit[2].items(): best_models.setdefault(nm, []).append((mdl, te_))
            print(f'  {task} rep{r} fold{j} cached', flush=True); continue
        n_rows, n_oof, fold_models = len(rows), len(oof), {}
        for nm, (est, space) in models.items():
            if space is None: mdl = est.fit(X[tr], y[tr])
            else:
                rs = RandomizedSearchCV(est, space, n_iter=NITER, cv=GroupKFold(KIN), scoring=scoring, random_state=SEED + 100 * r + j, n_jobs=1)
                mdl = rs.fit(X[tr], y[tr], groups=G[tr]).best_estimator_
            if task == 'clf':
                p = mdl.predict_proba(X[te_])[:, 1]
                rows.append(dict(rep=r, fold=j, model=nm, auc=roc_auc_score(y[te_], p), brier=brier_score_loss(y[te_], p), n_test=len(te_), n_train=len(tr)))
            else:
                p = mdl.predict(X[te_])
                rows.append(dict(rep=r, fold=j, model=nm, mae=mean_absolute_error(y[te_], p), r2=r2_score(y[te_], p), n_test=len(te_), n_train=len(tr)))
            oof += [dict(rep=r, fold=j, model=nm, idx=i, pred=pp) for i, pp in zip(te_, p)]
            if r == 0: best_models.setdefault(nm, []).append((mdl, te_)); fold_models[nm] = mdl
        if task == 'clf':
            dm = DummyClassifier(strategy='prior').fit(X[tr], y[tr]); p = dm.predict_proba(X[te_])[:, 1]
            rows.append(dict(rep=r, fold=j, model='Dummy', auc=.5, brier=brier_score_loss(y[te_], p), n_test=len(te_), n_train=len(tr)))
        else:
            dm = DummyRegressor().fit(X[tr], y[tr]); p = dm.predict(X[te_])
            rows.append(dict(rep=r, fold=j, model='Dummy', mae=mean_absolute_error(y[te_], p), r2=r2_score(y[te_], p), n_test=len(te_), n_train=len(tr)))
        cput(f'{task}_{r}_{j}', (rows[n_rows:], oof[n_oof:], fold_models))
        print(f'  {task} rep{r} fold{j} done', flush=True)
    return pd.DataFrame(rows), pd.DataFrame(oof), best_models

t0 = time.time()

In [ ]:
# ── 3. A1 ─────────────────────────────────────────────────────
X1 = A[A1_X].to_numpy(float); y1 = A.F.to_numpy()
F1, O1, M1 = nested('clf', X1, y1, CLF, 'roc_auc'); F1.to_csv(os.path.join(OUT, 'A1_folds.csv'), index=False)
ntr, nte = F1.n_train.mean(), F1.n_test.mean()
S1 = []
for nm, d in F1.groupby('model'):
    mu, lo, hi, p = nb_ci(d.auc - .5, nte, ntr) if nm != 'Dummy' else (0, 0, 0, 1)
    S1.append(dict(model=nm, auc=d.auc.mean(), lo=.5 + lo, hi=.5 + hi, brier=d.brier.mean(), n_folds=len(d), decision=('예측 가능(하한 > 0.5)' if .5 + lo > .5 else '예측 불가') if nm != 'Dummy' else '기준'))
S1 = pd.DataFrame(S1); S1.to_csv(os.path.join(OUT, 'A1_summary.csv'), index=False)
C1 = []
for nm in ['RF', 'HGB', 'MLP']:
    d = F1[F1.model == nm].sort_values(['rep', 'fold']).auc.to_numpy() - F1[F1.model == 'LR'].sort_values(['rep', 'fold']).auc.to_numpy()
    mu, lo, hi, p = nb_ci(d, nte, ntr); C1.append(dict(model=nm, vs='LR', diff=mu, lo=lo, hi=hi, p=p))
C1 = pd.DataFrame(C1)   # Holm은 아래 '# A1 C1 Holm'에서 계산
print('A1', round(time.time() - t0), flush=True)

In [ ]:
# ── 4. A3 ─────────────────────────────────────────────────────
k3 = A.o2p_max.notna(); A3 = A[k3].reset_index(drop=True); X3 = A3[A3_X].to_numpy(float); y3 = A3.o2p_max.to_numpy()
G_save, SPL_save = G, SPL; G = A3.subj.to_numpy(); SPL = outer_splits(G)
F3, O3, M3 = nested('reg', X3, y3, REG, 'neg_mean_absolute_error'); F3.to_csv(os.path.join(OUT, 'A3_folds.csv'), index=False)
ntr3, nte3 = F3.n_train.mean(), F3.n_test.mean()
S3 = pd.DataFrame([dict(model=nm, mae=d.mae.mean(), mae_lo=nb_ci(d.mae, nte3, ntr3)[1], mae_hi=nb_ci(d.mae, nte3, ntr3)[2], r2=d.r2.mean(), n_folds=len(d))
                   for nm, d in F3.groupby('model')]); S3.to_csv(os.path.join(OUT, 'A3_summary.csv'), index=False)
C3 = []
for nm in ['Ridge', 'RF', 'HGB', 'MLP']:
    d = F3[F3.model == nm].sort_values(['rep', 'fold']).mae.to_numpy() - F3[F3.model == 'Linear'].sort_values(['rep', 'fold']).mae.to_numpy()
    mu, lo, hi, p = nb_ci(d, nte3, ntr3); C3.append(dict(model=nm, vs='Linear', diff=mu, lo=lo, hi=hi, p=p))
C3 = pd.DataFrame(C3); order = np.argsort(C3.p.values); ph = np.empty(4); run = 0
for rank, i in enumerate(order): run = max(run, min(1, C3.p.values[i] * (4 - rank))); ph[i] = run
C3['p_holm'] = ph; C3['decision'] = np.where((C3.hi < 0) & (C3.p_holm < .05), 'ML이 선형보다 정확', '선형 대비 이득 근거 없음'); C3.to_csv(os.path.join(OUT, 'A3_compare.csv'), index=False)
best3 = S3[S3.model != 'Dummy'].sort_values('mae').model.iloc[0]
OO3 = O3[(O3.model == best3)].groupby('idx').pred.mean(); A3oof = A3[['id_test', 'subj', 'sex', 'age', 'o2p_max']].copy()
A3oof['pred'] = OO3.reindex(range(len(A3))).values; A3oof['resid'] = A3oof.o2p_max - A3oof.pred
lin = O3[O3.model == 'Linear'].groupby('idx').pred.mean(); A3oof['pred_linear'] = lin.reindex(range(len(A3))).values
A3oof.to_csv(os.path.join(OUT, 'A3_oof.csv'), index=False)
G, SPL = G_save, SPL_save
print('A3', round(time.time() - t0), flush=True)
# A1 C1 Holm(같은 방식으로 다시 — 위 한 줄식 대신 명시 계산)
order = np.argsort(C1.p.values); ph = np.empty(3); run = 0
for rank, i in enumerate(order): run = max(run, min(1, C1.p.values[i] * (3 - rank))); ph[i] = run
C1['p_holm'] = ph; C1['decision'] = np.where((C1.lo > 0) & (C1.p_holm < .05), 'ML 이득', 'ML 이득 근거 없음'); C1.to_csv(os.path.join(OUT, 'A1_compare.csv'), index=False)

In [ ]:
# ── 5. 하위군 · 중요도 ─────────────────────────────────────────
best1 = S1[S1.model != 'Dummy'].sort_values('auc', ascending=False).model.iloc[0]
oof1 = O1[(O1.model == best1) & (O1.rep == 0)].set_index('idx').pred; A['p_best'] = oof1.reindex(range(len(A))).values
A['agegrp'] = pd.cut(A.age, [18, 30, 40, 100], right=False, labels=['18–29', '30–39', '≥40']); A3oof['agegrp'] = pd.cut(A3oof.age, [18, 30, 40, 100], right=False, labels=['18–29', '30–39', '≥40'])
def boot(fn, D, nb=NB, seed=SEED):
    subj = D.subj.unique(); r = np.random.default_rng(seed); idx = {u: np.where(D.subj.values == u)[0] for u in subj}; vals = []
    for _ in range(nb):
        ii = np.concatenate([idx[u] for u in r.choice(subj, len(subj))])
        try: vals.append(fn(D.iloc[ii]))
        except ValueError: pass
    return np.percentile(vals, [2.5, 97.5])
SG = []
for nm, col, levs in [('sex', 'sex', [(0, 'Male'), (1, 'Female')]), ('age', 'agegrp', [(l, l) for l in ['18–29', '30–39', '≥40']])]:
    for lv, lab in levs:
        d = A[A[col] == lv]; f_ = lambda D: roc_auc_score(D.F, D.p_best); lo, hi = boot(f_, d)
        SG.append(dict(task='A1', metric=f'AUC ({best1})', stratifier=nm, level=lab, n=len(d), est=f_(d), lo=lo, hi=hi))
        d = A3oof[A3oof[col] == lv]; g_ = lambda D: mean_absolute_error(D.o2p_max, D.pred); lo, hi = boot(g_, d)
        SG.append(dict(task='A3', metric=f'MAE ({best3})', stratifier=nm, level=lab, n=len(d), est=g_(d), lo=lo, hi=hi))
SGD = pd.DataFrame(SG); SGD.to_csv(os.path.join(OUT, 'subgroup.csv'), index=False)
IMP = []
for task, X_, y_, mods, best, cols, sc in [('A1', X1, y1, M1, best1, A1_X, 'roc_auc'), ('A3', X3, y3, M3, best3, A3_X, 'neg_mean_absolute_error')]:
    acc = np.zeros(len(cols))
    for mdl, te_ in mods[best]:
        acc += permutation_importance(mdl, X_[te_], y_[te_], scoring=sc, n_repeats=10, random_state=SEED).importances_mean
    for c, v in zip(cols, acc / len(mods[best])): IMP.append(dict(task=task, model=best, feature=c, importance=v))
IMPD = pd.DataFrame(IMP); IMPD.to_csv(os.path.join(OUT, 'importance.csv'), index=False)
print('SG/PI', round(time.time() - t0), flush=True)

In [ ]:
# ── 6. A2 시계열 ───────────────────────────────────────────────
ids2 = list(A.id_test); STAT = A.set_index('id_test')[['age', 'sex', 'weight', 'height']]
TIERS = {'HR+speed': ['hr', 'sp'], 'HR+speed+resp': ['hr', 'sp', 'rr', 've']}
def lagfeat(sq, chans):
    cols = []
    for c in chans:
        x = sq[c]; cols.append(x)
        for w in (6, 12, 24): cols.append(pd.Series(x).rolling(w, min_periods=1).mean().to_numpy())   # 30 · 60 · 120 s 과거 평균(인과)
    cols.append(np.arange(len(sq['hr'])) * 5 / 60); return np.column_stack(cols)
def build(kind, cin):
    """v1과 같은 구조. CNN: 인과(왼쪽) 패딩 dilated Conv1D 4층 → 1×1 출력. GRU: 64 단위 → 시점별 Dense(1)."""
    x = inp = keras.Input((None, cin))
    if kind == 'CNN':
        for dl in (1, 2, 4, 8): x = layers.Conv1D(32, 5, dilation_rate=dl, padding='causal', activation='relu')(x)
        out = layers.Conv1D(1, 1)(x)
    else:
        out = layers.Dense(1)(layers.GRU(64, return_sequences=True)(x))
    return keras.Model(inp, out)
def mmse(yt, yp):
    """마스크 MSE: y_true[..., 0] = 표준화 VO2, y_true[..., 1] = 실제 시점 1 / 패딩 0 (v1 torch와 같은 식)."""
    m = yt[..., 1]; return ops.sum(ops.square(yp[..., 0] - yt[..., 0]) * m) / ops.maximum(ops.sum(m), 1.)
def arrays(idl, chans, mu, sd, ymu, ysd, smu, ssd):
    L = max(len(SEQ[i]['hr']) for i in idl); X = np.zeros((len(idl), L, len(chans) + 4 + 1), np.float32); Y = np.zeros((len(idl), L, 2), np.float32)
    for b, i in enumerate(idl):
        sq = SEQ[i]; n = len(sq['hr'])
        dyn = np.column_stack([sq[c] for c in chans] + [np.arange(n) * 5 / 60]); X[b, :n, :len(chans) + 1] = (dyn - mu) / sd
        X[b, :n, len(chans) + 1:] = ((STAT.loc[i].values - smu) / ssd)[None, :]; Y[b, :n, 0] = (sq['vo2'] - ymu) / ysd; Y[b, :n, 1] = 1
    return X, Y
def train_dl(kind, tr_ids, va_ids, chans):
    dyn = np.vstack([np.column_stack([SEQ[i][c] for c in chans] + [np.arange(len(SEQ[i]['hr'])) * 5 / 60]) for i in tr_ids])
    mu, sd = dyn.mean(0), dyn.std(0) + 1e-6; smu, ssd = STAT.loc[tr_ids].values.mean(0), STAT.loc[tr_ids].values.std(0) + 1e-6
    yy = np.concatenate([SEQ[i]['vo2'] for i in tr_ids]); ymu, ysd = yy.mean(), yy.std(); norm = (mu, sd, ymu, ysd, smu, ssd)
    Xtr, Ytr = arrays(tr_ids, chans, *norm); Xva, Yva = arrays(va_ids, chans, *norm)
    keras.backend.clear_session(); keras.utils.set_random_seed(SEED)
    net = build(kind, Xtr.shape[2]); net.compile(keras.optimizers.Adam(1e-3, global_clipnorm=1.0), loss=mmse)
    es = keras.callbacks.EarlyStopping('val_loss', patience=15, min_delta=1e-5, restore_best_weights=True)
    h = net.fit(Xtr, Ytr, batch_size=16, epochs=10 if FAST else 150, shuffle=True, validation_data=(Xva, Yva),
                validation_batch_size=len(Xva), callbacks=[es], verbose=0)   # 검증 손실 = 검증군 전체 한 묶음(v1과 같은 정의)
    return net, norm, len(h.history['loss'])
def predict_dl(net, norm, idl, chans):
    X, _ = arrays(idl, chans, *norm); P = net.predict(X, batch_size=64, verbose=0)[..., 0] * norm[3] + norm[2]
    return {i: P[b, :len(SEQ[i]['hr'])] for b, i in enumerate(idl)}
def last120_slope(vo2s, hrs, te):
    n = len(hrs); t = np.arange(n) * 5 + 2.5; k = t >= te - 120
    if k.sum() < 8: return np.nan
    b = ((t[k] - (te - 120)) // 10).astype(int); d = pd.DataFrame(dict(b=b, v=vo2s[k], h=hrs[k], t=t[k])).groupby('b').mean()
    return np.polyfit(d.t / 60, d.v / d.h, 1)[0]
# 측정 VO2를 같은 격자로 — 격자화 자체의 일치(천장)
A['slope_grid'] = [last120_slope(SEQ[i]['vo2'], SEQ[i]['hr'], SEQ[i]['te']) for i in ids2]; A['F_grid'] = (A.slope_grid <= 0).astype(int)
eng.append(dict(check='kappa F_grid vs F (5 s grid ceiling)', a=kappa(A.F_grid, A.F), b=np.nan)); E = pd.DataFrame(eng); E['absdiff'] = (E.a - E.b).abs()
subj2 = A.subj.to_numpy(); perm = np.random.default_rng(SEED).permutation(np.unique(subj2)); folds2 = np.array_split(perm, KOUT)
PRED = {}; EPOCHS = []
for tier, chans in TIERS.items():
    for kind in ['Ridge', 'CNN', 'GRU']:
        for j, fold in enumerate(folds2):
            hit = cget(f'a2_{tier}_{kind}_{j}')
            if hit is not None:
                PRED.update(hit[0]); EPOCHS += hit[1]; print(f'  A2 {tier} {kind} fold{j} cached', flush=True); continue
            n_ep = len(EPOCHS); keys_before = set(PRED)
            te_ids = [i for i, u in zip(ids2, subj2) if u in set(fold)]; tr_all = [i for i, u in zip(ids2, subj2) if u not in set(fold)]
            assert not set(A.set_index('id_test').loc[te_ids].subj) & set(A.set_index('id_test').loc[tr_all].subj)
            if kind == 'Ridge':
                Xtr = np.vstack([np.column_stack([lagfeat(SEQ[i], chans), np.repeat(STAT.loc[i].values[None], len(SEQ[i]['hr']), 0)]) for i in tr_all])
                ytr = np.concatenate([SEQ[i]['vo2'] for i in tr_all]); mdl = Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=1.0))]).fit(Xtr, ytr)
                for i in te_ids: PRED[(tier, kind, i)] = mdl.predict(np.column_stack([lagfeat(SEQ[i], chans), np.repeat(STAT.loc[i].values[None], len(SEQ[i]['hr']), 0)]))
            else:
                us = np.unique(A.set_index('id_test').loc[tr_all].subj); va_u = set(np.random.default_rng(SEED + j).choice(us, int(round(.15 * len(us))), replace=False))
                va_ids = [i for i in tr_all if A.set_index('id_test').at[i, 'subj'] in va_u]; tr_ids = [i for i in tr_all if i not in set(va_ids)]
                net, norm, ep = train_dl(kind, tr_ids, va_ids, chans); EPOCHS.append(dict(tier=tier, model=kind, fold=j, epochs=ep))
                PRED.update({(tier, kind, i): v for i, v in predict_dl(net, norm, te_ids, chans).items()})
            cput(f'a2_{tier}_{kind}_{j}', ({k_: PRED[k_] for k_ in set(PRED) - keys_before}, EPOCHS[n_ep:]))
            print(f'  A2 {tier} {kind} fold{j}', round(time.time() - t0), flush=True)
A2R = []; A2T = []
pidx = A.set_index('id_test')
for tier in TIERS:
    for kind in ['Ridge', 'CNN', 'GRU']:
        rows = []
        for i in ids2:
            sq = SEQ[i]; p = PRED[(tier, kind, i)]; n = len(p); t = np.arange(n) * 5 + 2.5; k = t >= sq['te'] - 120
            sl = last120_slope(p, sq['hr'], sq['te'])
            pk = pd.Series(p).rolling(6, min_periods=3).mean().max(); mk = pd.Series(sq['vo2']).rolling(6, min_periods=3).mean().max()
            rows.append(dict(tier=tier, model=kind, id_test=i, subj=pidx.at[i, 'subj'], F=pidx.at[i, 'F'], slope=pidx.at[i, 'main_slope'], slope_hat=sl, F_hat=int(sl <= 0),
                             mae_all=np.mean(np.abs(p - sq['vo2'])), mae_last=np.mean(np.abs(p[k] - sq['vo2'][k])), peak_err=pk - mk))
        D = pd.DataFrame(rows); A2T.append(D)
        kp = kappa(D.F_hat, D.F); lo, hi = boot(lambda X: kappa(X.F_hat, X.F), D)
        A2R.append(dict(tier=tier, model=kind, kappa=kp, lo=lo, hi=hi, agree=np.mean(D.F_hat == D.F), prev_hat=D.F_hat.mean(), n=len(D),
                        vo2_mae=D.mae_all.mean(), vo2_mae_last120=D.mae_last.mean(), slope_mae=np.mean(np.abs(D.slope_hat - D.slope)),
                        peak_err_mean=D.peak_err.mean(), peak_err_sd=D.peak_err.std(),
                        decision='가스분석 없이 판정 가능(하한 ≥ 0.40)' if lo >= .4 else ('불가(상한 < 0.40)' if hi < .4 else '판정 불가')))
A2S = pd.DataFrame(A2R); A2S.to_csv(os.path.join(OUT, 'A2_summary.csv'), index=False)
A2D = pd.concat(A2T, ignore_index=True); A2D.to_csv(os.path.join(OUT, 'A2_per_test.csv'), index=False)    # 로컬 전용(DUA)
pd.DataFrame(EPOCHS).to_csv(os.path.join(OUT, 'A2_epochs.csv'), index=False)
print('A2', round(time.time() - t0), flush=True)

In [ ]:
# ── 7. A4 군집 ─────────────────────────────────────────────────
def shape(tid):
    sq = SEQ[tid]; n = len(sq['hr']); b = np.minimum((np.arange(n) / n * 20).astype(int), 19)
    d = pd.DataFrame(dict(b=b, v=sq['vo2'], h=sq['hr'])).groupby('b').mean(); o = (d.v / d.h).reindex(range(20)).interpolate(limit_direction='both').to_numpy()
    return o / np.nanmax(o)
A['shape'] = [shape(i) for i in ids2]
F4 = A.sort_values(['subj', 'age', 'k']).drop_duplicates('subj').reset_index(drop=True); X4 = np.vstack(F4['shape'].values)
BIC = []; GM = {}
for k in range(1, 7):
    gm = GaussianMixture(k, covariance_type='diag', n_init=10, random_state=SEED).fit(X4); GM[k] = gm; BIC.append(dict(k=k, bic=gm.bic(X4)))
    if k >= 2: BIC[-1]['silhouette_kmeans'] = silhouette_score(X4, KMeans(k, n_init=10, random_state=SEED).fit_predict(X4))
BICD = pd.DataFrame(BIC); BICD.to_csv(os.path.join(OUT, 'A4_bic.csv'), index=False); kbest = int(BICD.loc[BICD.bic.idxmin(), 'k'])
# 사후(○): 탐색 경계(k = 6)에서 최소 → k 1–12로 넓혀 확인 [핸즈온1 p.130–133 '최적값이 격자 경계면 범위를 넓혀 다시']
PHA = [dict(analysis='A4 BIC k', k=k, value=GaussianMixture(k, covariance_type='diag', n_init=10, random_state=SEED).fit(X4).bic(X4)) for k in range(1, 13)]
# 사후(○): speed_slope(직전 2분 속도 증가율)가 중요도 1위 → 램프 속도 범주별 F 비율(프로토콜 교란 점검)
A['ramp_pre'] = pd.cut(A.speed_slope, [-np.inf, .98, 1.02, np.inf], labels=['< 0.98 km/h/min', '0.98–1.02 (standard)', '> 1.02'])
for lv, d in A.groupby('ramp_pre', observed=True): PHA.append(dict(analysis='F by ramp', k=str(lv), value=d.F.mean(), n=len(d)))
PHAD = pd.DataFrame(PHA); PHAD.to_csv(os.path.join(OUT, 'posthoc_ai.csv'), index=False)
gm = GM[kbest]; F4['cluster'] = gm.predict(X4)
A4C = pd.DataFrame([dict(cluster=c, n=int((F4.cluster == c).sum()), prop_F=F4[F4.cluster == c].F.mean(), **{f'b{b}': gm.means_[c, b] for b in range(20)}) for c in range(kbest)])
A4C.to_csv(os.path.join(OUT, 'A4_clusters.csv'), index=False)
pairs = []
for u, g in A.sort_values(['subj', 'age', 'k']).groupby('subj'):
    if len(g) >= 2: pairs.append(dict(subj=u, c1=int(gm.predict(g['shape'].iloc[0][None])[0]), c2=int(gm.predict(g['shape'].iloc[1][None])[0])))
PR = pd.DataFrame(pairs)
if kbest >= 2 and len(PR) >= 2:
    kp4 = kappa(PR.c1, PR.c2); r_ = np.random.default_rng(SEED); bs = [kappa(*PR.iloc[r_.integers(0, len(PR), len(PR))][['c1', 'c2']].values.T) for _ in range(NB)]
    lo4, hi4 = np.nanpercentile(bs, [2.5, 97.5])
else: kp4, lo4, hi4 = np.nan, np.nan, np.nan
A4R = pd.DataFrame([dict(k_best=kbest, n_first=len(F4), n_pairs=len(PR), kappa=kp4, lo=lo4, hi=hi4, agree=np.mean(PR.c1 == PR.c2) if len(PR) else np.nan,
                         decision='표현형 없음(k = 1)' if kbest == 1 else ('재현되는 표현형(하한 ≥ 0.40)' if lo4 >= .4 else ('재현 안 됨(상한 < 0.40)' if hi4 < .4 else '판정 불가')))])
A4R.to_csv(os.path.join(OUT, 'A4_repeat.csv'), index=False)
print('A4', round(time.time() - t0), flush=True)

In [ ]:
# ── 8. 이상치(⑫ 타당 범위 + 강건 z 1회, 표시만) ─────────────────
OL = []; nA = len(A3)
for c in ['o2p_max', 'hrmax', 'hr_at12', 'hr_speed_slope']:
    x = A3[c]; med = x.median(); mad = (x - med).abs().median(); M = .6745 * (x - med) / mad
    for i in np.where(M.abs() > 3.5)[0]: OL.append(dict(id_test=A3.id_test[i], variable=c, value=x[i], rule='robust z |M|>3.5', expected_false_pos=nA * 2 * stats.norm.sf(3.5), handling='표시만'))
for c in ['o2p_max', 'hrmax']:
    a, b = VALID[c]; x = A3[c]
    for i in np.where((x < a) | (x > b))[0]: OL.append(dict(id_test=A3.id_test[i], variable=c, value=x[i], rule=f'valid {a}–{b}', expected_false_pos=np.nan, handling='표시만'))
OLD = pd.DataFrame(OL, columns=['id_test', 'variable', 'value', 'rule', 'expected_false_pos', 'handling']); OLD.to_csv(os.path.join(OUT, 'outliers.csv'), index=False)
E.to_csv(os.path.join(OUT, 'engine_check.csv'), index=False)
FL = pd.DataFrame([dict(step='per_test rows (o2p_v3)', n=len(T)), dict(step='excluded (218_1, 741_1, 539_1)', n=int((T.fail.fillna('') != '').sum())),
                   dict(step='age < 18', n=int(((T.fail.fillna('') == '') & (T.age < 18)).sum())),
                   dict(step='not evaluable (120 s window)', n=int(((T.fail.fillna('') == '') & (T.age >= 18) & (T.main_evaluable != True)).sum())),
                   dict(step='A1 / A2 tests', n=len(A)), dict(step='A1 / A2 participants', n=A.subj.nunique()), dict(step='A3 tests (O2P max available)', n=len(A3)),
                   dict(step='A4 first tests', n=len(F4)), dict(step='A4 repeat pairs', n=len(PR))])
FL.to_csv(os.path.join(OUT, 'sample_flow.csv'), index=False)

In [ ]:
# ── 9. 그림(§7 묶음 · §8 양식) ─────────────────────────────────
from matplotlib.ticker import NullLocator
plt.rcParams.update({'font.size': 9, 'font.family': 'DejaVu Sans', 'axes.edgecolor': '#222222', 'axes.linewidth': .8, 'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})
BLUE, ORANGE, GREY, NAVY = '#2a78d6', '#eb6834', '#8a8a8a', '#1b4f9c'; jr = np.random.default_rng(JSEED)
def gg(ax, grid='y'): ax.set_axisbelow(True); ax.grid(axis=grid, color='#ececec', lw=.35) if grid else None
def ttl(ax, t, sub=None):
    ax.set_title(t, loc='left', fontweight='bold', fontsize=9.5, pad=14 if sub else 6)
    if sub: ax.text(0, 1.01, sub, transform=ax.transAxes, fontsize=7.5, color='#666666', va='bottom')
def save(fig, nm):
    fig.savefig(os.path.join(FIG, nm + '.png'), dpi=200, bbox_inches='tight')
    if display: display(fig)
    plt.close(fig)
def dots(ax, j, y, c):
    y = np.asarray(y, float); vp = ax.violinplot(y, [j], widths=.7, showextrema=False)
    for b in vp['bodies']: b.set_facecolor('#dfe7f2'); b.set_alpha(1)
    ax.scatter(j + jr.uniform(-.12, .12, len(y)), y, s=8, color=c, lw=0, zorder=3)
# G1
fig, ax = plt.subplots(figsize=(6.5, 3.8)); ax.axis('off')
for i, r in FL.iterrows(): ax.text(.02, .92 - i * .1, f'{r.step}: {r.n}', transform=ax.transAxes, fontsize=9)
ttl(ax, 'G1 Sample flow — AI tasks', 'o2p_v3 per-test table; participant-grouped CV uses all adult evaluable tests'); save(fig, 'G1_flow')
# G2
fig, ax = plt.subplots(figsize=(5, 3.4)); gg(ax, 'x'); Ee = E[E.b.notna()]
ax.scatter(Ee.absdiff + 1e-17, range(len(Ee)), s=12, color=NAVY); ax.set_xscale('log'); ax.set_yticks(range(len(Ee)), Ee.check, fontsize=6.5); ax.xaxis.set_minor_locator(NullLocator())
kg = E[E.check.str.startswith('kappa F_grid')].a.iloc[0]; ax.set_xlabel('|this script − reference| (log; 1e-17 = exact)')
ttl(ax, 'G2 Engine checks', f'all < 1e-10 (assert); 5 s grid vs breath-level primary F: κ = {kg:.2f} (ceiling for A2)'); save(fig, 'G2_engine')
# G3
fig, axs = plt.subplots(1, 3, figsize=(16, 4.4), gridspec_kw=dict(width_ratios=[1.1, 1.1, 1.2], wspace=.6)); g3 = []
ax = axs[0]; gg(ax); mods = ['Dummy', 'LR', 'RF', 'HGB', 'MLP']
for j, nm in enumerate(mods):
    d = F1[F1.model == nm]; dots(ax, j, d.auc, GREY if nm in ('Dummy', 'LR') else BLUE); r = S1[S1.model == nm].iloc[0]
    if nm != 'Dummy': ax.errorbar([j + .3], [r.auc], yerr=[[r.auc - r.lo], [r.hi - r.auc]], fmt='D', color='k', ms=4, capsize=2); ax.text(j + .33, r.hi + .01, f'{r.auc:.2f}', fontsize=7)
    g3 += [dict(panel='A1', model=nm, value=v) for v in d.auc]
ax.axhline(.5, color=ORANGE, ls='--', lw=.7); ax.set_xticks(range(len(mods)), mods); ax.set_ylabel('AUC (outer fold)')
ttl(ax, 'a  A1 early prediction of flattening', f'{len(SPL)} outer folds per model; ◆ mean, Nadeau–Bengio 95% CI; n={len(A)} tests')
ax = axs[1]; gg(ax); mods3 = ['Dummy', 'Linear', 'Ridge', 'RF', 'HGB', 'MLP']
for j, nm in enumerate(mods3):
    d = F3[F3.model == nm]; dots(ax, j, d.mae, GREY if nm in ('Dummy', 'Linear') else BLUE); r = S3[S3.model == nm].iloc[0]
    ax.errorbar([j + .3], [r.mae], yerr=[[r.mae - r.mae_lo], [r.mae_hi - r.mae]], fmt='D', color='k', ms=4, capsize=2); ax.text(j + .33, r.mae_hi + .03, f'{r.mae:.2f}', fontsize=7)
    g3 += [dict(panel='A3', model=nm, value=v) for v in d.mae]
ax.set_xticks(range(len(mods3)), mods3, fontsize=8); ax.set_ylabel('MAE, peak O2 pulse (mL/beat)'); ttl(ax, 'b  A3 peak O2 pulse without gas analysis', f'{len(SPL)} outer folds per model; ◆ mean, Nadeau–Bengio 95% CI; n={len(A3)} tests')
ax = axs[2]; gg(ax, 'x'); lab = []
for i, r in A2S.reset_index(drop=True).iterrows():
    ax.errorbar([r.kappa], [-i], xerr=[[r.kappa - r.lo], [r.hi - r.kappa]], fmt='o', color=BLUE if r.tier == 'HR+speed' else NAVY, capsize=2); lab.append(f'{r.model} · {r.tier}')
    g3.append(dict(panel='A2', model=f'{r.model}|{r.tier}', value=r.kappa))
ax.axvline(.4, color=ORANGE, ls='--', lw=.7); ax.axvline(kg, color=GREY, ls=':', lw=.8); ax.set_xlim(-.05, 1)
ax.set_yticks(-np.arange(len(A2S)), lab, fontsize=7.5); ax.set_xlabel('κ, flattening from estimated vs measured O2 pulse')
ttl(ax, 'c  A2 flattening without gas analysis', f'n={len(A)} tests; participant bootstrap 95% CI; dotted = 5 s grid ceiling κ {kg:.2f}')
G3D = pd.DataFrame(g3); assert (G3D.panel == 'A1').sum() == len(F1) and (G3D.panel == 'A3').sum() == len(F3); G3D.to_csv(os.path.join(FIG, 'G3_data.csv'), index=False)
fig.suptitle('G3 Primary results — AI tasks', x=.01, y=1.05, ha='left', fontweight='bold'); save(fig, 'G3_primary')
# G4
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw=dict(wspace=.4))
for ax, task, xl in [(axs[0], 'A1', 'AUC'), (axs[1], 'A3', 'MAE (mL/beat)')]:
    gg(ax, 'x'); d = SGD[SGD.task == task].reset_index(drop=True)
    for i, r in d.iterrows(): ax.errorbar([r.est], [-i], xerr=[[r.est - r.lo], [r.hi - r.est]], fmt='o', color='k', capsize=2); pass
    ax.set_yticks(-np.arange(len(d)), [f'{r.stratifier}: {r.level} (n={r.n})' for _, r in d.iterrows()], fontsize=8); ax.set_xlabel(xl); ttl(ax, f'{task} — {d.metric.iloc[0]}', ('OOF, repeat 1' if task == 'A1' else 'OOF, mean of 5 repeats') + '; participant bootstrap CI')
fig.suptitle('G4 Subgroup performance', x=.01, y=1.08, ha='left', fontweight='bold'); save(fig, 'G4_subgroups')
# G5
fig, axs = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw=dict(wspace=.45))
for ax, C, xl, ref in [(axs[0], C1, 'ΔAUC vs logistic', 'LR'), (axs[1], C3, 'ΔMAE vs linear (mL/beat; < 0 = better)', 'Linear')]:
    gg(ax, 'x')
    for i, r in C.reset_index(drop=True).iterrows(): ax.errorbar([r['diff']], [-i], xerr=[[r['diff'] - r.lo], [r.hi - r['diff']]], fmt='o', color='k', capsize=2)
    ax.axvline(0, color=GREY, lw=.8); ax.set_yticks(-np.arange(len(C)), [f'● {m}  (P_Holm ' + ('< 0.001' if p < .001 else f'{p:.3f}') + ')' for m, p in zip(C.model, C.p_holm)], fontsize=8); ax.set_xlabel(xl)
fig.suptitle('G5 Model comparisons (● pre-specified; Nadeau–Bengio corrected 95% CI)', x=.01, y=1.05, ha='left', fontweight='bold'); save(fig, 'G5_compare')
# G6
fig, axs = plt.subplots(1, 2, figsize=(11, 4.4), gridspec_kw=dict(wspace=.5))
for ax, task in [(axs[0], 'A1'), (axs[1], 'A3')]:
    gg(ax, 'x'); d = IMPD[IMPD.task == task].sort_values('importance'); ax.barh(range(len(d)), d.importance, color=BLUE); ax.set_yticks(range(len(d)), d.feature, fontsize=7)
    ax.axvline(0, color=GREY, lw=.8); ax.set_xlabel('Permutation importance (drop in AUC)' if task == 'A1' else 'Permutation importance (rise in MAE)'); ttl(ax, f'{task} — {d.model.iloc[0]}', 'outer test folds of repeat 1, 10 permutations')
fig.suptitle('G6 Feature importance (block decomposition not applicable to prediction models)', x=.01, y=1.06, ha='left', fontweight='bold'); save(fig, 'G6_importance')
# G7
fig, axs = plt.subplots(1, 2, figsize=(10, 3.4), gridspec_kw=dict(wspace=.35))
for ax, col, xl in [(axs[0], 'mae_all', 'VO2 MAE, whole exercise (mL/min)'), (axs[1], 'mae_last', 'VO2 MAE, last 120 s (mL/min)')]:
    gg(ax); j = 0; ticks = []
    for kind in ['Ridge', 'CNN', 'GRU']:
        for tier, c in [('HR+speed', BLUE), ('HR+speed+resp', NAVY)]:
            d = A2D[(A2D.model == kind) & (A2D.tier == tier)][col]; dots(ax, j, d, c); ax.errorbar([j + .3], [d.mean()], yerr=[[Z * d.std() / np.sqrt(len(d))]], fmt='D', color='k', ms=3, capsize=2)
            ticks.append(f'{kind}\n{"HR+sp" if tier == "HR+speed" else "+resp"}'); j += 1
    ax.set_xticks(range(j), ticks, fontsize=7); ax.set_ylabel(xl); ttl(ax, xl.split(',')[0] + (' — whole' if col == 'mae_all' else ' — last 120 s'), f'each point = one test (n={len(A)})')
fig.suptitle('G7 A2 input tiers (blue = HR + speed, navy = + respiratory rate and VE)', x=.01, y=1.06, ha='left', fontweight='bold'); save(fig, 'G7_a2_inputs')
# G8
best2 = A2S.sort_values('kappa', ascending=False).iloc[0]; fig, axs = plt.subplots(1, 3, figsize=(15, 3.8), gridspec_kw=dict(wspace=.4)); g8 = []
for fv, c in [(0, BLUE), (1, ORANGE)]:
    ms, ps = [], []
    for i in A[A.F == fv].id_test:
        sq = SEQ[i]; p = PRED[(best2.tier, best2.model, i)]; n = len(p); k = np.arange(n) * 5 + 2.5 >= sq['te'] - 120
        mo = (sq['vo2'] / sq['hr'])[k][-24:]; po = (p / sq['hr'])[k][-24:]
        if len(mo) == 24: ms.append(mo); ps.append(po)
    ms, ps = np.array(ms), np.array(ps); x = np.arange(24) * 5 - 117.5
    axs[0].plot(x, ms.mean(0), color=c, lw=1.8, label=f'measured, F={fv} (n={len(ms)})'); axs[0].plot(x, ps.mean(0), color=c, lw=1.2, ls='--', label=f'estimated, F={fv}')
    g8 += [dict(panel='A2', F=fv, t=tt, measured=a_, estimated=b_) for tt, a_, b_ in zip(x, ms.mean(0), ps.mean(0))]
gg(axs[0]); axs[0].set_xlabel('Time before exercise end (s)'); axs[0].set_ylabel('O2 pulse (mL/beat)'); axs[0].legend(fontsize=6.5, loc='lower left')
ttl(axs[0], f'a  A2 {best2.model} · {best2.tier}', 'means: measured (solid) vs estimated (dashed)')
ax = axs[1]; gg(ax)
for c_ in range(kbest):
    d = F4[F4.cluster == c_]; Xs = np.vstack(d['shape'].values)
    for row in Xs[:150]: ax.plot(np.arange(20) * 5 + 2.5, row, color='#e3e3e3', lw=.3)
for c_, col in zip(range(kbest), [BLUE, ORANGE, NAVY, GREY, '#6b4bb3', '#2e9e6a']):
    d = F4[F4.cluster == c_]; Xs = np.vstack(d['shape'].values); ax.plot(np.arange(20) * 5 + 2.5, Xs.mean(0), color=col, lw=1.8, label=f'cluster {c_} (n={len(d)}, F {100*d.F.mean():.0f}%)')
    g8 += [dict(panel='A4', cluster=c_, t=tt, mean_shape=v) for tt, v in zip(np.arange(20) * 5 + 2.5, Xs.mean(0))]
ax.set_xlabel('Exercise time (% of duration)'); ax.set_ylabel('O2 pulse / own maximum'); ax.legend(fontsize=6.5, loc='lower right'); ttl(ax, f'b  A4 shape clusters (BIC k = {kbest})', 'grey = up to 150 tests per cluster')
ax = axs[2]; gg(ax); ax.plot(BICD.k, BICD.bic, 'o-', color=NAVY); ax.set_xlabel('k'); ax.set_ylabel('BIC'); r4 = A4R.iloc[0]
ph4 = PHAD[PHAD.analysis == 'A4 BIC k']; ax.plot(ph4.k, ph4.value, 'o--', mfc='white', color=GREY, zorder=1, label='○ post hoc k 1–12')
ax.legend(fontsize=7, loc='upper right'); ttl(ax, 'c  A4 BIC (pre-specified k 1–6: min at k = 6, the boundary)', f'repeat pairs n={int(r4.n_pairs)}: κ {r4.kappa:.2f} [{r4.lo:.2f}, {r4.hi:.2f}]; post hoc BIC min at k={int(ph4.loc[ph4.value.idxmin(), "k"])}')
pd.DataFrame(g8).to_csv(os.path.join(FIG, 'G8_data.csv'), index=False); fig.suptitle('G8 Curves', x=.01, y=1.06, ha='left', fontweight='bold'); save(fig, 'G8_curves')
# G9
fig, axs = plt.subplots(1, 3, figsize=(13, 3.6), gridspec_kw=dict(wspace=.35))
ax = axs[0]; gg(ax, 'both'); ax.scatter(A3oof.pred, A3oof.resid, s=5, color=NAVY, alpha=.6); ax.axhline(0, color=GREY, lw=.8)
ax.set_xlabel('Predicted peak O2 pulse (mL/beat)'); ax.set_ylabel('Residual'); ttl(ax, f'a  A3 {best3} residuals', f'out-of-fold, n={len(A3oof)}')
ax = axs[1]; gg(ax, 'both'); (osm, osr), (sl_, ic_, _) = stats.probplot(A3oof.resid.dropna(), dist='norm'); ax.scatter(osm, osr, s=5, color=NAVY); ax.plot(osm, sl_ * osm + ic_, color=GREY, lw=.8)
ax.set_xlabel('Normal quantile'); ax.set_ylabel('Residual quantile'); ttl(ax, 'b  Q–Q')
ax = axs[2]; gg(ax); j = 0; ticks = []
for kind in ['Ridge', 'CNN', 'GRU']:
    for tier, c in [('HR+speed', BLUE), ('HR+speed+resp', NAVY)]:
        d = A2D[(A2D.model == kind) & (A2D.tier == tier)].peak_err; dots(ax, j, d, c); ticks.append(f'{kind}\n{"HR+sp" if tier == "HR+speed" else "+resp"}'); j += 1
ax.axhline(0, color=GREY, lw=.8); ax.set_xticks(range(j), ticks, fontsize=7); ax.set_ylabel('VO2peak error, estimated − measured (mL/min)'); ttl(ax, 'c  A2 peak VO2 error', f'n={len(A)} per model')
assert A3oof.pred.notna().sum() == len(A3oof); A3oof.to_csv(os.path.join(FIG, 'G9_data.csv'), index=False)
fig.suptitle('G9 Residual diagnostics', x=.01, y=1.06, ha='left', fontweight='bold'); save(fig, 'G9_residuals')
# G10
fig, axs = plt.subplots(1, 4, figsize=(13, 3.6), gridspec_kw=dict(wspace=.45)); g10 = []
for ax, c in zip(axs, ['o2p_max', 'hrmax', 'hr_at12', 'hr_speed_slope']):
    gg(ax); x = A3[c].to_numpy(float); k = ~np.isnan(x); vp = ax.violinplot(x[k], [0], widths=.8, showextrema=False)
    for b in vp['bodies']: b.set_facecolor('#dfe7f2'); b.set_alpha(1)
    xx = jr.uniform(-.15, .15, len(x)); ax.scatter(xx, x, s=4, color=GREY, lw=0)
    fl = set(OLD[(OLD.variable == c)].id_test); mk = A3.id_test.isin(fl).values; ax.scatter(xx[mk], x[mk], marker='x', color=ORANGE, s=25)
    if c in VALID: [ax.axhline(v, color=GREY, lw=.6, ls=':') for v in VALID[c]]
    ax.set_xticks([]); ttl(ax, c, f'n={k.sum()} · flagged {mk.sum()}'); g10 += [dict(variable=c, id_test=t_, value=v_) for t_, v_ in zip(A3.id_test, x)]
assert len(g10) == 4 * len(A3); pd.DataFrame(g10).to_csv(os.path.join(FIG, 'G10_data.csv'), index=False)
fig.suptitle('G10 Outlier screen of A3 inputs and target (flagged only)', x=.01, y=1.08, ha='left', fontweight='bold'); save(fig, 'G10_outliers')

In [ ]:
# ── 10. v1 대조 · 판정 · 실행 정보
# A1 · A3 · A4 · A2 Ridge = 같은 코드·seed·입력. 수치 동일은 '같은 실행 환경(패키지 버전)'에서만 기대한다.
# 다른 PC · 버전(예: scikit-learn 1.8 → 1.9)에서는 랜덤 서치 · 모형 내부 계산이 달라 수치가 움직인다
#   → 그때는 판정(decision)이 같은지로 확인한다(PI 재실행 2026-10-22에서 확인된 사항).
# A2 CNN/GRU = 프레임워크 변경(torch → Keras) → 차이만 기록.
VC, DEC = [], []
if os.path.isdir(V1_RES):
    for fn, key, cols in [('A1_summary.csv', ['model'], ['auc', 'lo', 'hi', 'brier']), ('A1_compare.csv', ['model'], ['diff', 'lo', 'hi', 'p_holm']),
                          ('A3_summary.csv', ['model'], ['mae', 'mae_lo', 'mae_hi', 'r2']), ('A3_compare.csv', ['model'], ['diff', 'lo', 'hi', 'p_holm']),
                          ('A4_repeat.csv', ['k_best'], ['kappa', 'lo', 'hi']), ('A2_summary.csv', ['tier', 'model'], ['kappa', 'lo', 'hi', 'vo2_mae', 'vo2_mae_last120', 'slope_mae'])]:
        a_ = pd.read_csv(os.path.join(V1_RES, fn)); b_ = pd.read_csv(os.path.join(OUT, fn)); mm = a_.merge(b_, on=key, suffixes=('_v1', '_v2'))
        if len(mm) != len(a_):   # A4 k_best가 달라 병합이 안 되는 경우 → 판정 비교에서 잡는다
            print(f'⚠️ {fn}: 키 {key}가 v1과 달라 수치 대조 생략'); continue
        for _, r in mm.iterrows():
            for c in cols:
                dl_ = fn == 'A2_summary.csv' and r.get('model') in ('CNN', 'GRU')
                VC.append(dict(file=fn, row='|'.join(str(r[k]) for k in key), col=c, v1=r[c + '_v1'], v2=r[c + '_v2'], diff=r[c + '_v2'] - r[c + '_v1'],
                               expect='differs (framework)' if dl_ else 'identical (same environment only)'))
    VCD = pd.DataFrame(VC)
    bad = VCD[(VCD.expect != 'differs (framework)') & ~(VCD['diff'].abs() < 1e-9)]
    if bad.empty: print('비 딥러닝 수치: v1과 동일(차 < 1e-9)')
    else: print(f'⚠️ 비 딥러닝 수치가 v1과 다르다({len(bad)}칸, 최대 차 {bad["diff"].abs().max():.3g}) — 실행 환경(패키지 버전) 차이로 본다. 판정 비교로 확인.')
    # 판정 비교(환경과 무관하게 같아야 한다)
    for fn, key in [('A1_summary.csv', ['model']), ('A1_compare.csv', ['model']), ('A3_compare.csv', ['model']), ('A2_summary.csv', ['tier', 'model'])]:
        mm = pd.read_csv(os.path.join(V1_RES, fn)).merge(pd.read_csv(os.path.join(OUT, fn)), on=key, suffixes=('_v1', '_v2'))
        DEC += [dict(file=fn, row='|'.join(str(r[k]) for k in key), v1=r.decision_v1, v2=r.decision_v2) for _, r in mm.iterrows()]
    a4, b4 = pd.read_csv(os.path.join(V1_RES, 'A4_repeat.csv')).iloc[0], pd.read_csv(os.path.join(OUT, 'A4_repeat.csv')).iloc[0]
    DEC.append(dict(file='A4_repeat.csv', row=f'k {a4.k_best}→{b4.k_best}', v1=a4.decision, v2=b4.decision))
    DECD = pd.DataFrame(DEC); DECD['same'] = DECD.v1 == DECD.v2
    DECD.to_csv(os.path.join(OUT, 'v1_vs_v2_decisions.csv'), index=False); print(DECD.to_string(index=False))
    assert DECD.same.all(), f'판정이 v1과 달라졌다:\n{DECD[~DECD.same]}'
else: VCD = pd.DataFrame([dict(file='(v1 결과 폴더 없음)', row='', col='', v1=np.nan, v2=np.nan, diff=np.nan, expect='skipped')])
VCD.to_csv(os.path.join(OUT, 'v1_vs_v2.csv'), index=False)
dec = dict(A1=dict(best=best1, summary=S1.to_dict('records'), compare=C1.to_dict('records')),
           A2=A2S.to_dict('records'), A3=dict(best=best3, summary=S3.to_dict('records'), compare=C3.to_dict('records')), A4=A4R.to_dict('records'))
json.dump(dec, open(os.path.join(OUT, 'decisions.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1, default=float)
pk = ['numpy', 'pandas', 'scipy', 'scikit-learn', 'tensorflow', 'keras', 'matplotlib']
def ver(p):
    for q in (p, p + '-cpu', p.replace('tensorflow', 'tensorflow_cpu'), p.replace('tensorflow', 'tensorflow-intel')):
        try: return metadata.version(q)
        except metadata.PackageNotFoundError: pass
    return 'not found'
open(os.path.join(OUT, 'requirements_exact.txt'), 'w', encoding='utf-8').write('\n'.join(f'{p}=={ver(p)}' for p in pk) + f'\n# python {platform.python_version()} · {platform.platform()}\n')
json.dump(dict(generation='o2p_ai_v2', framework=f'keras {keras.__version__} / tensorflow {tf.__version__}', platform=platform.platform(), python=platform.python_version(),
               fast_debug=FAST, run_at=datetime.datetime.now().isoformat(timespec='seconds'), seed=SEED, repeats=REPEATS, k_outer=KOUT, k_inner=KIN,
               n_iter=NITER, sha256=SHA, per_test=PERTEST, per_test_sha256=PT_SHA, seconds=round(time.time() - t0),
               v1_numeric_identical=bool(os.path.isdir(V1_RES) and bad.empty), v1_decisions_identical=bool(len(DEC) and DECD.same.all())),
          open(os.path.join(OUT, 'run_info.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
missing = [f for f in EXPECTED_OUTPUTS if not os.path.exists(os.path.join(OUT, f))]; assert not missing, f'정본 산출물 누락: {missing}'
print(S1.round(3).to_string(index=False)); print(C1.round(3).to_string(index=False)); print(S3.round(3).to_string(index=False)); print(C3.round(3).to_string(index=False))
print(A2S.round(3).to_string(index=False)); print(A4R.round(3).to_string(index=False)); print(BICD.round(1).to_string(index=False)); print('seconds', round(time.time() - t0))